# ILPCSR Embedding Gap & Chunk Coverage Analysis

Diagnoses which ILPCSR corpus cases / paragraphs / chunks are missing from NeonDB and **why**,
then defines and verifies the canonical "complete" end state.

> **Read the narrative first:** `notebooks/ilpcsr_embedding_gap_findings.md` summarizes the root
> cause, the full numbers chain (711 -> 616 + 95 -> 30), and the runbook. This notebook is the
> evidence behind that doc -- run the cells to reproduce every number.

## Reading guide

| Section | Cells | Status |
|---|---|---|
| **1-6 - Case-level gap** (corpus vs `ilpcsr_cases`) -- found 22 never-ingested (9 bad dates + missing summaries) | `gap0002`-`gap0014` | RESOLVED -> 3183/3183 |
| **7 - Post-ingestion completeness check** (5 quick counts) | `e5043fbc`, `3689bde8` | Superseded by Sec 8 -- kept for context |
| **7a - Orphan-paragraph exploration** (711 orphans; strict twin tests) | `8abc373a`-`4d2e3b44` | **Exploratory journey** -- these use over-strict / wrong-grain definitions that produced misleading numbers (137 / 574 / 30-at-paragraph-grain). Kept as an audit trail; do NOT treat their verdicts as final. See Sec 8. |
| **8 - Canonical end-state definition & verification** | `d4bd7525`-`45f4c5fe` | **Authoritative.** The 3 checks here + the content-grain coverage queries are the source of truth. |

**Bottom line:** the only real gap is **30 distinct chunk-texts with zero embeddings**. Everything
else (711 orphan paragraphs, 363 duplicate chunk groups, "574 uncovered") is an *allowed* duplicate
or an artifact of an over-strict check. See Sec 8 and the findings doc.

## Original section outline (1-6)
1. Setup - 2. Corpus inventory - 3. DB inventory - 4. Gap computation - 5. Failure diagnosis - 6. Summary

## 1 · Setup

In [ ]:
import os
import json
import datetime
import psycopg2
import psycopg2.extras
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

conn = psycopg2.connect(
    dbname=os.environ['DB_NAME'],
    user=os.environ['DB_USER'],
    password=os.environ['DB_PASSWORD'],
    host=os.environ['DB_HOST'],
    port=os.environ['DB_PORT'],
    sslmode='require'
)

CORPUS_DIR = os.path.join('..', 'il-pcsr_dataset', 'corpus')
EMB_MODEL_ID = 1

print('Connected    :', conn.status == psycopg2.extensions.STATUS_READY)
print('Corpus dir   :', os.path.abspath(CORPUS_DIR))
print('Emb model ID :', EMB_MODEL_ID)

## 2 · Corpus inventory

Derives case IDs from filenames only — no JSON reads here.

In [ ]:
corpus_ids = {
    fname[:-5]  # strip .json
    for fname in os.listdir(CORPUS_DIR)
    if fname.endswith('.json')
}

print(f'Corpus cases : {len(corpus_ids)}')

## 3 · DB inventory

In [ ]:
cur = conn.cursor()

cur.execute('SELECT doc_id FROM ilpcsr_cases')
db_case_ids = {row[0] for row in cur.fetchall()}

cur.execute(
    'SELECT doc_id FROM ilpcsr_cases_embeddings WHERE emb_model_id = %s',
    (EMB_MODEL_ID,)
)
embedding_ids = {row[0] for row in cur.fetchall()}

cur.close()

print(f'In ilpcsr_cases            : {len(db_case_ids)}')
print(f'In ilpcsr_cases_embeddings : {len(embedding_ids)}')

## 4 · Gap computation

In [ ]:
never_ingested = corpus_ids - db_case_ids      # text phase never wrote these
orphaned       = db_case_ids - corpus_ids       # in DB but no corpus file (sanity check)

print(f'Never ingested (corpus \ DB) : {len(never_ingested)}')
print(f'Orphaned in DB (DB \ corpus) : {len(orphaned)}')

if orphaned:
    print('\nOrphaned IDs (unexpected):')
    for cid in sorted(orphaned)[:20]:
        print(f'  {cid}')
else:
    print('\nNo orphaned IDs — DB is a subset of corpus. ✓')

print(f'\nSample never_ingested (up to 10):')
for cid in sorted(never_ingested)[:10]:
    print(f'  {cid}')

## 4b · Atomicity check — cases with text but no embeddings

For cases in `ilpcsr_cases` but missing from `ilpcsr_cases_embeddings`, checks whether their text data is fully present (paragraphs + chunks) or partially committed. Expected: either fully present text (embedding phase failed after text commit) or nothing (which would put them in `never_ingested` instead).

In [ ]:
text_only = db_case_ids - embedding_ids

print(f"Cases in ilpcsr_cases but no ilpcsr_cases_embeddings row : {len(text_only)}")

if not text_only:
    print("All ingested cases have embeddings — fully clean. ✓")
else:
    text_only_list = list(text_only)

    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    cur.execute("""
        SELECT
            c.doc_id,
            COUNT(DISTINCT p.paragraph_id)  AS para_count,
            COUNT(DISTINCT cc.chunk_id)     AS chunk_count,
            COUNT(DISTINCT ce.chunk_emb_id) AS chunk_emb_count
        FROM ilpcsr_cases c
        LEFT JOIN ilpcsr_paragraphs p      ON p.doc_id      = c.doc_id
        LEFT JOIN ilpcsr_content_chunks cc ON cc.paragraph_id = p.paragraph_id
        LEFT JOIN ilpcsr_chunk_embeddings ce ON ce.chunk_id  = cc.chunk_id
        WHERE c.doc_id = ANY(%s)
        GROUP BY c.doc_id
    """, (text_only_list,))
    rows = cur.fetchall()
    cur.close()

    atomicity_df = pd.DataFrame(rows)

    no_paras   = atomicity_df[atomicity_df['para_count']      == 0]
    no_chunks  = atomicity_df[(atomicity_df['para_count']     >  0) & (atomicity_df['chunk_count']    == 0)]
    no_chembs  = atomicity_df[(atomicity_df['chunk_count']    >  0) & (atomicity_df['chunk_emb_count'] == 0)]
    full_text  = atomicity_df[atomicity_df['chunk_emb_count'] >  0]

    print(f"\n  Has paragraphs + chunks + chunk_embeddings (embedding phase failed at case-level only) : {len(full_text)}")
    print(f"  Has paragraphs + chunks but NO chunk embeddings                                        : {len(no_chembs)}")
    print(f"  Has paragraphs but NO chunks                                                           : {len(no_chunks)}")
    print(f"  In ilpcsr_cases but NO paragraphs at all (partial case-row only)                       : {len(no_paras)}")

    if not no_paras.empty:
        print(f"\n⚠️  Partial case rows (no paragraphs) — unexpected:")
        print(no_paras[['doc_id']].to_string(index=False))

    if not no_chunks.empty:
        print(f"\n⚠️  Cases with paragraphs but no chunks — unexpected:")
        print(no_chunks[['doc_id', 'para_count']].to_string(index=False))

    print(f"\nFull breakdown (first 20 rows):")
    print(atomicity_df.sort_values('chunk_count').head(20).to_string(index=False))

## 5 · Failure diagnosis for `never_ingested`

Reads only the JSON files for cases in `never_ingested`. Checks:
- `caseID` present and matches filename
- `Summary` present and non-empty (all 5 fields)
- `date` parseable as a PostgreSQL-compatible date string
- Any other top-level anomalies

`ingestion/logs/` does not exist (no prior local ingestion run) — all diagnosis is from JSON content.

In [ ]:
DATE_FORMATS = [
    '%Y-%m-%d',
    '%d/%m/%Y',
    '%d-%m-%Y',
    '%B %d, %Y',
    '%d %B %Y',
    '%b %d, %Y',
    '%d %b %Y',
    '%Y/%m/%d',
]

SUMMARY_FIELDS = ['key_facts', 'legal_issues', 'key_statutes_cited', 'key_precedents_cited', 'holding']

def parse_date(raw):
    if not raw:
        return None, 'date_missing'
    for fmt in DATE_FORMATS:
        try:
            datetime.datetime.strptime(raw.strip(), fmt)
            return fmt, None
        except ValueError:
            pass
    return None, 'bad_date_format'

results = []

if not never_ingested:
    print('never_ingested is empty — all corpus cases are in the DB. ✓')
else:
    for case_id in sorted(never_ingested):
        fpath = os.path.join(CORPUS_DIR, f'{case_id}.json')
        row = {
            'case_id'         : case_id,
            'file_exists'     : False,
            'caseid_match'    : None,
            'has_summary'     : None,
            'summary_complete': None,
            'date_raw'        : None,
            'date_issue'      : None,
            'date_parsed_as'  : None,
            'other_issues'    : [],
        }

        if not os.path.exists(fpath):
            row['other_issues'].append('file_not_found')
            results.append(row)
            continue

        row['file_exists'] = True

        try:
            with open(fpath, 'r', encoding='utf-8') as f:
                data = json.load(f)
        except Exception as e:
            row['other_issues'].append(f'json_parse_error:{e}')
            results.append(row)
            continue

        cid_in_file = str(data.get('caseID', ''))
        row['caseid_match'] = (cid_in_file == case_id)
        if not cid_in_file:
            row['other_issues'].append('missing_caseID')

        summary = data.get('Summary')
        row['has_summary'] = summary is not None
        if summary is None:
            row['other_issues'].append('missing_Summary_key')
            row['summary_complete'] = False
        else:
            def _is_empty(v):
                if v is None: return True
                if isinstance(v, str): return not v.strip()
                if isinstance(v, list): return len(v) == 0
                return False
            blank_fields = [f for f in SUMMARY_FIELDS if _is_empty(summary.get(f))]
            row['summary_complete'] = len(blank_fields) == 0
            if blank_fields:
                row['other_issues'].append(f'blank_summary_fields:{blank_fields}')

        raw_date = data.get('date', '')
        row['date_raw'] = raw_date
        parsed_fmt, issue = parse_date(raw_date)
        row['date_issue']     = issue
        row['date_parsed_as'] = parsed_fmt

        results.append(row)

    df = pd.DataFrame(results)
    df['other_issues'] = df['other_issues'].apply(lambda x: '; '.join(x) if x else '')
    print(f'Cases diagnosed: {len(df)}')

In [ ]:
# Breakdown by failure category
if not results:
    print('No never_ingested cases — skipping diagnosis breakdown. ✓')
else:
    print('=== Failure breakdown ===')
    print(f"  bad_date_format           : {(df['date_issue'] == 'bad_date_format').sum()}")
    print(f"  date_missing (not failure): {(df['date_issue'] == 'date_missing').sum()}")
    print(f"  missing Summary key       : {(df['has_summary'] == False).sum()}")
    print(f"  summary present but blank : {((df['has_summary'] == True) & (df['summary_complete'] == False)).sum()}")
    print(f"  caseID mismatch           : {(df['caseid_match'] == False).sum()}")
    print(f"  json parse error / missing: {df['other_issues'].str.contains('json_parse_error|file_not_found').sum()}")
    print()

    bad_date = df[df['date_issue'] == 'bad_date_format'][['case_id', 'date_raw']]
    if not bad_date.empty:
        print('=== bad_date_format samples (up to 20) ===')
        print(bad_date.head(20).to_string(index=False))
    else:
        print('No bad date format issues found.')

In [ ]:
if not results:
    print('No never_ingested cases — skipping summary diagnosis. ✓')
else:
    missing_summary = df[(df['has_summary'] == False) | (df['summary_complete'] == False)]
    if not missing_summary.empty:
        print('=== Missing / incomplete Summary (up to 20) ===')
        print(missing_summary[['case_id', 'has_summary', 'summary_complete', 'other_issues']].head(20).to_string(index=False))
    else:
        print('All never_ingested cases have complete Summaries.')
    print()

    other = df[df['other_issues'].str.contains('json_parse_error|file_not_found|missing_caseID|caseID_mismatch', na=False)]
    if not other.empty:
        print('=== Other anomalies ===')
        print(other[['case_id', 'other_issues']].to_string(index=False))

## 6 · Summary

In [ ]:
print('=' * 50)
print('ILPCSR Embedding Gap — Summary')
print('=' * 50)
print(f'  Corpus total                  : {len(corpus_ids)}')
print(f'  In ilpcsr_cases               : {len(db_case_ids)}')
print(f'  In ilpcsr_cases_embeddings    : {len(embedding_ids)}')
print(f'  Never ingested (text gap)     : {len(never_ingested)}')
print(f'  Orphaned in DB                : {len(orphaned)}')
print()
print('Never-ingested root causes:')
if not df.empty:
    print(f'  bad_date_format    : {(df["date_issue"] == "bad_date_format").sum()}')
    print(f'  missing Summary    : {(df["has_summary"] == False).sum()}')
    print(f'  incomplete Summary : {((df["has_summary"] == True) & (df["summary_complete"] == False)).sum()}')
    print(f'  caseID mismatch    : {(df["caseid_match"] == False).sum()}')
    print(f'  other / unknown    : {df["other_issues"].str.contains("json_parse_error|file_not_found").sum()}')
else:
    print('  (no cases to diagnose)')
print('=' * 50)

conn.close()
print('Connection closed.')

## 7 - Post-ingestion completeness check

> **Note:** This was an early 5-count check. Two of its rows ("paragraphs without chunks",
> "same-para duplicate groups") flag *allowed duplicates* as failures and are therefore
> misleading. **Section 8 supersedes this** with the correct content-grain definition.
> Run this after `embed_missing_chunks.py` + a full `ingest_ilpcsr_to_db.py` pass.

All five checks must pass before the corpus is considered complete.

| Check | Expectation |
|---|---|
| Cases in DB | 3183 |
| Case-level embeddings | 3183 |
| Paragraphs with no chunks | 0 |
| Same-paragraph duplicate chunks | 0 |
| Eligible unembedded chunks remaining | 0 |

## 7a - Orphan paragraph exploration  (EXPLORATORY - superseded by Section 8)

> **Audit trail, not conclusions.** This block investigates the 711 paragraphs with no chunks.
> Several cells here use **over-strict or wrong-grain** coverage definitions and produced numbers
> that turned out to be artifacts:
> - `ffadf04d`, `d66f13ef` -- "fully embedded twin" test (per-paragraph `emb_count = chunk_count`);
>   counts allowed Rule-B duplicate chunks as failures -> reports 137 covered / **574** "uncovered".
> - `4e83527d` -- invariant at `(paragraph_id, chunk_text)` grain -> 30 (right idea, wrong grain).
> - `4d2e3b44` -- "reachability picture" using the same over-strict twin test.
>
> The **correct** content-grain analysis is in **Section 8** (`cc4e631d`, `45f4c5fe`):
> 711 orphans = 616 already covered + **95** real gaps, and those 95 collapse to the **30**
> zero-embedding content groups. See `ilpcsr_embedding_gap_findings.md`.

Investigates the 711 paragraphs with no chunks -- how many cases are affected, whether the orphan
paragraphs share their `full_text` with another paragraph in the same case (the dict-collision
bug), and what labels are most affected.

In [ ]:
import os, psycopg2, psycopg2.extras, pandas as pd
from dotenv import load_dotenv
load_dotenv()

_conn = psycopg2.connect(
    dbname=os.environ['DB_NAME'], user=os.environ['DB_USER'],
    password=os.environ['DB_PASSWORD'], host=os.environ['DB_HOST'],
    port=os.environ['DB_PORT'], sslmode='require'
)
_cur = _conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

# ── All paragraphs with no chunks ────────────────────────────────────────────
_cur.execute("""
    SELECT p.paragraph_id, p.doc_id, p.metadata_label,
           LENGTH(p.full_text) AS text_len
    FROM   ilpcsr_paragraphs p
    LEFT JOIN ilpcsr_content_chunks cc ON cc.paragraph_id = p.paragraph_id
    WHERE  cc.chunk_id IS NULL
""")
orphans = pd.DataFrame(_cur.fetchall())
print(f"Orphan paragraphs (no chunks): {len(orphans)}")
print(f"Distinct cases affected      : {orphans['doc_id'].nunique()}")
print()

# ── Label distribution ────────────────────────────────────────────────────────
print("By metadata_label:")
print(orphans['metadata_label'].value_counts().to_string())
print()

# ── Are they empty / near-empty text? ────────────────────────────────────────
print("full_text length distribution:")
print(orphans['text_len'].describe().to_string())
empty = orphans[orphans['text_len'] == 0]
print(f"\nActually empty (len=0) : {len(empty)}")
print()

# ── Do they have a sibling with identical full_text in the same case? ─────────
# This would confirm the full_text_to_pid dict-collision bug in the ingest script.
_cur.execute("""
    SELECT COUNT(*) AS cnt
    FROM (
        SELECT p.paragraph_id
        FROM   ilpcsr_paragraphs p
        LEFT JOIN ilpcsr_content_chunks cc ON cc.paragraph_id = p.paragraph_id
        WHERE  cc.chunk_id IS NULL
          AND  EXISTS (
               SELECT 1 FROM ilpcsr_paragraphs p2
               WHERE  p2.doc_id    = p.doc_id
                 AND  p2.full_text = p.full_text
                 AND  p2.paragraph_id != p.paragraph_id
          )
    ) t
""")
with_twin = _cur.fetchone()['cnt']
print(f"Orphan paragraphs that have a full_text twin in the same case : {with_twin}")
print(f"Orphan paragraphs with NO twin (truly isolated)               : {len(orphans) - with_twin}")

_cur.close()
_conn.close()

In [ ]:
import os, psycopg2, psycopg2.extras
from dotenv import load_dotenv
load_dotenv()

_conn = psycopg2.connect(
    dbname=os.environ['DB_NAME'], user=os.environ['DB_USER'],
    password=os.environ['DB_PASSWORD'], host=os.environ['DB_HOST'],
    port=os.environ['DB_PORT'], sslmode='require'
)
_cur = _conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

EMB_MODEL_ID = 1

# Per orphan: does ANY of its twins have chunks AND all those chunks embedded?
_cur.execute("""
    WITH orphans AS (
        SELECT p.paragraph_id AS orphan_pid, p.doc_id, p.full_text
        FROM   ilpcsr_paragraphs p
        LEFT JOIN ilpcsr_content_chunks cc ON cc.paragraph_id = p.paragraph_id
        WHERE  cc.chunk_id IS NULL
    ),
    twin_stats AS (
        SELECT o.orphan_pid,
               COUNT(cc.chunk_id)     AS chunk_count,
               COUNT(ce.chunk_emb_id) AS emb_count
        FROM   orphans o
        JOIN   ilpcsr_paragraphs p2
               ON  p2.doc_id      = o.doc_id
               AND p2.full_text   = o.full_text
               AND p2.paragraph_id != o.orphan_pid
        LEFT JOIN ilpcsr_content_chunks cc  ON cc.paragraph_id = p2.paragraph_id
        LEFT JOIN ilpcsr_chunk_embeddings ce
               ON  ce.chunk_id     = cc.chunk_id
               AND ce.emb_model_id = %s
        GROUP BY o.orphan_pid, p2.paragraph_id
    ),
    per_orphan AS (
        SELECT orphan_pid,
               BOOL_OR(chunk_count > 0 AND emb_count = chunk_count) AS has_covered_twin
        FROM   twin_stats
        GROUP BY orphan_pid
    )
    SELECT
        COUNT(*)                                        AS total_orphans,
        COUNT(*) FILTER (WHERE has_covered_twin)        AS fully_covered,
        COUNT(*) FILTER (WHERE NOT has_covered_twin)    AS not_covered
    FROM per_orphan
""", (EMB_MODEL_ID,))

row = _cur.fetchone()
print("=== Twin keeper verification for 711 orphan paragraphs ===")
print(f"  Total orphans                         : {row['total_orphans']}")
print(f"  Covered (twin has chunks + embeddings): {row['fully_covered']}")
print(f"  NOT covered ⚠                         : {row['not_covered']}")
print()

if row['not_covered'] == 0:
    print("PASS ✓ — every orphan has a chunked + fully embedded twin. Content fully covered.")
else:
    print("FAIL ✗ — some orphans have no covered twin. Those paragraphs' content is unreachable.")

_cur.close()
_conn.close()

In [ ]:
import os
import psycopg2
from dotenv import load_dotenv

load_dotenv()

_conn = psycopg2.connect(
    dbname=os.environ['DB_NAME'],
    user=os.environ['DB_USER'],
    password=os.environ['DB_PASSWORD'],
    host=os.environ['DB_HOST'],
    port=os.environ['DB_PORT'],
    sslmode='require'
)
_cur = _conn.cursor()

CORPUS_TOTAL  = 3183
EMB_MODEL_ID  = 1
results       = {}

# ── CHECK 1: total cases ─────────────────────────────────────────────────────
_cur.execute("SELECT COUNT(*) FROM ilpcsr_cases")
results['cases_in_db'] = _cur.fetchone()[0]

# ── CHECK 2: case-level embeddings ───────────────────────────────────────────
_cur.execute(
    "SELECT COUNT(*) FROM ilpcsr_cases_embeddings WHERE emb_model_id = %s",
    (EMB_MODEL_ID,)
)
results['cases_embedded'] = _cur.fetchone()[0]

# ── CHECK 3: cases in ilpcsr_cases with no embedding row ────────────────────
_cur.execute("""
    SELECT c.doc_id FROM ilpcsr_cases c
    LEFT JOIN ilpcsr_cases_embeddings ce
           ON ce.doc_id = c.doc_id AND ce.emb_model_id = %s
    WHERE ce.case_embedding_id IS NULL
""", (EMB_MODEL_ID,))
missing_case_embs = [r[0] for r in _cur.fetchall()]

# ── CHECK 4: paragraphs with no chunks ──────────────────────────────────────
_cur.execute("""
    SELECT COUNT(*) FROM ilpcsr_paragraphs p
    LEFT JOIN ilpcsr_content_chunks cc ON cc.paragraph_id = p.paragraph_id
    WHERE cc.chunk_id IS NULL
""")
results['paras_without_chunks'] = _cur.fetchone()[0]

# ── CHECK 5: same-paragraph duplicate chunks ─────────────────────────────────
_cur.execute("""
    SELECT COUNT(*) FROM (
        SELECT paragraph_id, chunk_text
        FROM   ilpcsr_content_chunks
        GROUP  BY paragraph_id, chunk_text
        HAVING COUNT(*) > 1
    ) dups
""")
results['same_para_dup_groups'] = _cur.fetchone()[0]

# ── CHECK 6: eligible unembedded chunks (no same-para twin, no embedding) ────
_cur.execute("""
    SELECT COUNT(*)
    FROM   ilpcsr_content_chunks cc
    LEFT JOIN ilpcsr_chunk_embeddings ce
           ON ce.chunk_id = cc.chunk_id AND ce.emb_model_id = %s
    WHERE  ce.chunk_emb_id IS NULL
      AND  NOT EXISTS (
           SELECT 1 FROM ilpcsr_content_chunks cc2
           WHERE  cc2.paragraph_id = cc.paragraph_id
             AND  cc2.chunk_text   = cc.chunk_text
             AND  cc2.chunk_id    != cc.chunk_id
      )
""", (EMB_MODEL_ID,))
results['eligible_unembedded_chunks'] = _cur.fetchone()[0]

_cur.close()
_conn.close()

# ── Print results ─────────────────────────────────────────────────────────────
CHECKS = [
    ('Cases in DB',                   'cases_in_db',                CORPUS_TOTAL, lambda a, e: a == e),
    ('Case-level embeddings',         'cases_embedded',             CORPUS_TOTAL, lambda a, e: a == e),
    ('Paragraphs without chunks',     'paras_without_chunks',       0,            lambda a, e: a == e),
    ('Same-para duplicate groups',    'same_para_dup_groups',       0,            lambda a, e: a == e),
    ('Eligible unembedded chunks',    'eligible_unembedded_chunks', 0,            lambda a, e: a == e),
]

print(f"{'Check':<36} {'Expected':>10} {'Actual':>10}  Status")
print('-' * 65)
all_pass = True
for label, key, expected, ok in CHECKS:
    actual = results[key]
    status = 'PASS ✓' if ok(actual, expected) else 'FAIL ✗'
    if not ok(actual, expected):
        all_pass = False
    print(f"{label:<36} {expected:>10} {actual:>10}  {status}")

print('-' * 65)
print('ALL CHECKS PASSED ✓' if all_pass else 'SOME CHECKS FAILED — see rows marked FAIL ✗')

if missing_case_embs:
    print(f"\nCases missing embeddings ({len(missing_case_embs)}):")
    for cid in missing_case_embs[:20]:
        print(f"  {cid}")

In [ ]:
import os, psycopg2, psycopg2.extras
from dotenv import load_dotenv
load_dotenv()

_conn = psycopg2.connect(
    dbname=os.environ['DB_NAME'], user=os.environ['DB_USER'],
    password=os.environ['DB_PASSWORD'], host=os.environ['DB_HOST'],
    port=os.environ['DB_PORT'], sslmode='require'
)
_cur = _conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

# Core invariant: for every (paragraph_id, chunk_text) pair,
# at least one chunk_id must have an embedding.
# Groups with zero embedded copies = invariant violated.
_cur.execute("""
    SELECT cc.paragraph_id::text, cc.chunk_text,
           COUNT(cc.chunk_id)     AS total_copies,
           COUNT(ce.chunk_emb_id) AS embedded_copies
    FROM   ilpcsr_content_chunks cc
    LEFT JOIN ilpcsr_chunk_embeddings ce
           ON  ce.chunk_id     = cc.chunk_id
           AND ce.emb_model_id = 1
    GROUP  BY cc.paragraph_id, cc.chunk_text
    HAVING COUNT(ce.chunk_emb_id) = 0
""")
violations = _cur.fetchall()

print(f"(paragraph_id, chunk_text) groups with ZERO embeddings: {len(violations)}")
if violations:
    print("\nSample violating groups (up to 10):")
    for v in violations[:10]:
        print(f"  para={v['paragraph_id']}  copies={v['total_copies']}  text={repr(v['chunk_text'][:80])}")
    print("\nFAIL ✗ — invariant violated. These groups need one copy embedded.")
else:
    print("\nPASS ✓ — every (paragraph_id, chunk_text) has at least one embedded copy.")

_cur.close()
_conn.close()

In [ ]:
import os, psycopg2, psycopg2.extras
from dotenv import load_dotenv
load_dotenv()

_conn = psycopg2.connect(
    dbname=os.environ['DB_NAME'], user=os.environ['DB_USER'],
    password=os.environ['DB_PASSWORD'], host=os.environ['DB_HOST'],
    port=os.environ['DB_PORT'], sslmode='require'
)
_cur = _conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

EMB_MODEL_ID = 1

# ── A: paragraphs with NO chunks ─────────────────────────────────────────────
_cur.execute("""
    SELECT COUNT(*) AS cnt FROM ilpcsr_paragraphs p
    LEFT JOIN ilpcsr_content_chunks cc ON cc.paragraph_id = p.paragraph_id
    WHERE cc.chunk_id IS NULL
""")
no_chunks = _cur.fetchone()['cnt']

# ── B: of those, how many have a twin para with ≥1 embedded chunk ────────────
_cur.execute("""
    SELECT COUNT(*) AS cnt FROM (
        SELECT p.paragraph_id
        FROM   ilpcsr_paragraphs p
        LEFT JOIN ilpcsr_content_chunks cc ON cc.paragraph_id = p.paragraph_id
        WHERE  cc.chunk_id IS NULL
          AND  EXISTS (
               SELECT 1
               FROM   ilpcsr_paragraphs p2
               JOIN   ilpcsr_content_chunks cc2  ON cc2.paragraph_id = p2.paragraph_id
               JOIN   ilpcsr_chunk_embeddings ce ON ce.chunk_id = cc2.chunk_id
                                                AND ce.emb_model_id = %s
               WHERE  p2.doc_id      = p.doc_id
                 AND  p2.full_text   = p.full_text
                 AND  p2.paragraph_id != p.paragraph_id
          )
    ) t
""", (EMB_MODEL_ID,))
no_chunks_but_twin_covered = _cur.fetchone()['cnt']

# ── C: paragraphs with chunks but ≥1 (para,text) group has 0 embeddings ──────
_cur.execute("""
    SELECT COUNT(DISTINCT cc.paragraph_id) AS cnt
    FROM   ilpcsr_content_chunks cc
    LEFT JOIN ilpcsr_chunk_embeddings ce
           ON  ce.chunk_id     = cc.chunk_id
           AND ce.emb_model_id = %s
    GROUP  BY cc.paragraph_id, cc.chunk_text
    HAVING COUNT(ce.chunk_emb_id) = 0
""", (EMB_MODEL_ID,))
paras_with_unembedded_groups = _cur.fetchone()['cnt'] if _cur.rowcount else 0

# ── D: total paragraphs ───────────────────────────────────────────────────────
_cur.execute("SELECT COUNT(*) AS cnt FROM ilpcsr_paragraphs")
total_paras = _cur.fetchone()['cnt']

_cur.close()
_conn.close()

print("=== Full reachability picture ===")
print(f"  Total paragraphs in DB                          : {total_paras}")
print()
print(f"  Paragraphs with NO chunks                       : {no_chunks}")
print(f"    → of those, twin para with embedded chunk     : {no_chunks_but_twin_covered}  (content reachable via twin)")
print(f"    → truly unreachable (no twin coverage)        : {no_chunks - no_chunks_but_twin_covered}")
print()
print(f"  Paragraphs with chunks but a (para,text) group")
print(f"    has zero embeddings (invariant violation)     : {paras_with_unembedded_groups}")
print()
truly_unreachable = (no_chunks - no_chunks_but_twin_covered) + paras_with_unembedded_groups
print(f"  TOTAL paragraphs with unreachable content       : {truly_unreachable}")
if truly_unreachable == 0:
    print("\n  PASS ✓ — all paragraph content is reachable via at least one embedded chunk.")
else:
    print("\n  FAIL ✗ — some content is genuinely unreachable.")

# 8 · Canonical End-State Definition & Verification

> **This section supersedes the ad-hoc checks above.** Earlier cells measured *"do unembedded
> copies / duplicate rows exist?"* — but duplicates are **allowed**. The only thing that matters
> is *"is every piece of content embedded at least once?"*

## Required end state

1. **All 3183 cases have case-level embeddings** — `ilpcsr_cases_embeddings` count = 3183.
2. **All case text is present as paragraphs** — every case's text lives in `ilpcsr_paragraphs`.
3. **Every distinct piece of content is chunked + embedded exactly once.**

## What is allowed to be skipped (NOT a failure)

Only two kinds of exact char-for-char duplicates may go un-embedded — **provided one copy of
each is already chunked and embedded**:

- **Rule A:** A paragraph whose `full_text` is char-for-char identical to another paragraph
  **in the same case** (`doc_id`). One copy must be chunked + embedded; the other(s) may be skipped.
- **Rule B:** A chunk whose `chunk_text` is char-for-char identical to another chunk
  **in the same paragraph**. One copy must be embedded; the other(s) may be skipped.

The **union of both rules** = grouping content at the grain **`(doc_id, full_text, chunk_text)`**.
At this grain, twin paragraphs (Rule A) and same-paragraph duplicate chunks (Rule B) collapse
into a single content group. The invariant is: **every such group must have ≥ 1 embedded copy.**

## The two definitive checks

- **CHECK 1 — no content is entirely chunk-less:** every `(doc_id, full_text)` group must have
  at least one paragraph that has chunks. (Catches the blind spot: an all-orphan group whose
  content exists nowhere as a chunk.)
- **CHECK 2 — every content is embedded once:** group all chunks by `(doc_id, full_text, chunk_text)`;
  the count of groups with **zero** embedded copies must be **0**.

## Why earlier numbers were misleading

| Earlier number | What it actually was |
|---|---|
| 711 paragraphs without chunks | Rule A duplicates — allowed, *if* a twin is embedded (CHECK 1 confirms) |
| 363 same-para duplicate groups | Rule B duplicates — allowed, *if* one copy is embedded (CHECK 2 confirms) |
| "574 not covered" | Artifact of an over-strict `emb_count == chunk_count` test that flagged the *allowed* state as failure |
| "30 violations" | Measured at `(paragraph_id, chunk_text)` grain; at content grain it is ≤ 30 — these are the only real fixes |

**Root cause (for future re-ingestion):** `full_text_to_pid` at line 178 of
`ingestion/ingest_ilpcsr_to_db.py` is keyed by `full_text` — same dict-collision bug as the
already-fixed `chunk_text_to_cid`. It is the origin of both the 711 orphans and the 363 dup groups.
Re-ingesting on a fresh DB with the current code **deterministically reproduces this same state**.
Fix line 178 (composite key) before any clean re-ingest.

In [ ]:
import os, time, psycopg2, psycopg2.extras
from dotenv import load_dotenv
load_dotenv()

_conn = psycopg2.connect(
    dbname=os.environ['DB_NAME'], user=os.environ['DB_USER'],
    password=os.environ['DB_PASSWORD'], host=os.environ['DB_HOST'],
    port=os.environ['DB_PORT'], sslmode='require'
)
_conn.autocommit = True   # no lingering transaction / locks
_cur = _conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
_cur.execute("SET statement_timeout = '120s'")   # fail fast instead of hanging
EMB_MODEL_ID = 1
print("connected, timeout set", flush=True)

# ── CHECK 0 ──────────────────────────────────────────────────────────────────
t = time.time()
_cur.execute("SELECT COUNT(*) AS c FROM ilpcsr_cases_embeddings WHERE emb_model_id = %s", (EMB_MODEL_ID,))
cases_embedded = _cur.fetchone()['c']
print(f"CHECK 0 done in {time.time()-t:.1f}s  -> cases_embedded={cases_embedded}", flush=True)

# ── CHECK 1: content groups (hashed) with NO chunk anywhere ──────────────────
# Plain LEFT JOIN aggregation, hash the text key. No correlated subquery.
t = time.time()
_cur.execute("""
    SELECT p.doc_id, md5(p.full_text) AS ft_hash,
           COUNT(cc.chunk_id) AS chunk_count
    FROM   ilpcsr_paragraphs p
    LEFT JOIN ilpcsr_content_chunks cc ON cc.paragraph_id = p.paragraph_id
    GROUP  BY p.doc_id, md5(p.full_text)
    HAVING COUNT(cc.chunk_id) = 0
""")
chunkless_content = _cur.fetchall()
print(f"CHECK 1 done in {time.time()-t:.1f}s  -> chunkless groups={len(chunkless_content)}", flush=True)

# ── CHECK 2: content groups (hashed) with ZERO embeddings ────────────────────
t = time.time()
_cur.execute("""
    SELECT p.doc_id, md5(p.full_text) AS ft_hash, md5(cc.chunk_text) AS ct_hash,
           COUNT(cc.chunk_id)     AS total_copies,
           COUNT(ce.chunk_emb_id) AS embedded_copies,
           MIN(cc.chunk_text)     AS sample_text
    FROM   ilpcsr_content_chunks cc
    JOIN   ilpcsr_paragraphs p ON p.paragraph_id = cc.paragraph_id
    LEFT JOIN ilpcsr_chunk_embeddings ce
           ON  ce.chunk_id     = cc.chunk_id
           AND ce.emb_model_id = %s
    GROUP  BY p.doc_id, md5(p.full_text), md5(cc.chunk_text)
    HAVING COUNT(ce.chunk_emb_id) = 0
""", (EMB_MODEL_ID,))
unembedded_content = _cur.fetchall()
print(f"CHECK 2 done in {time.time()-t:.1f}s  -> zero-embedding groups={len(unembedded_content)}", flush=True)

_cur.close()
_conn.close()

# ── Report ────────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("CANONICAL END-STATE VERIFICATION  (content grain: doc_id, full_text, chunk_text)")
print("=" * 70)

c0 = cases_embedded == 3183
print(f"\nCHECK 0  Cases embedded                    : {cases_embedded:>5} / 3183   {'PASS ✓' if c0 else 'FAIL ✗'}")

c1 = len(chunkless_content) == 0
print(f"CHECK 1  (doc_id, full_text) with NO chunks : {len(chunkless_content):>5}          {'PASS ✓' if c1 else 'FAIL ✗'}")
if not c1:
    print("         ⚠ content groups with NO chunked copy anywhere (unreachable):")
    for r in chunkless_content[:10]:
        print(f"           doc={r['doc_id']}  ft_hash={r['ft_hash']}")

c2 = len(unembedded_content) == 0
print(f"CHECK 2  content groups w/ ZERO embeddings  : {len(unembedded_content):>5}          {'PASS ✓' if c2 else 'FAIL ✗'}")
if not c2:
    print("         ⚠ exist as chunks but NONE embedded — need 1 each:")
    for r in unembedded_content[:15]:
        print(f"           doc={r['doc_id']}  copies={r['total_copies']}  text={repr((r['sample_text'] or '')[:70])}")

print("\n" + "-" * 70)
if c0 and c1 and c2:
    print("ALL CHECKS PASSED ✓ — corpus is in the required end state.")
else:
    print("NOT in end state. Fix: embed one representative chunk per CHECK-2 group")
    print("(pure INSERTs, no deletes). CHECK 1 failures (if any) need re-ingest.")

In [ ]:
import os, time, psycopg2, psycopg2.extras
from dotenv import load_dotenv
load_dotenv()

_conn = psycopg2.connect(
    dbname=os.environ['DB_NAME'], user=os.environ['DB_USER'],
    password=os.environ['DB_PASSWORD'], host=os.environ['DB_HOST'],
    port=os.environ['DB_PORT'], sslmode='require'
)
_conn.autocommit = True
_cur = _conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
_cur.execute("SET statement_timeout = '120s'")
EMB_MODEL_ID = 1
print("connected", flush=True)

# For every orphan paragraph (no chunks), require a twin paragraph in the SAME case
# with EXACT char-for-char identical full_text (p2.full_text = p.full_text, no hashing)
# whose chunks are ALL embedded (chunk_count > 0 AND emb_count = chunk_count).
t = time.time()
_cur.execute("""
    WITH orphan AS (
        SELECT p.paragraph_id AS orphan_pid, p.doc_id, p.full_text
        FROM   ilpcsr_paragraphs p
        LEFT JOIN ilpcsr_content_chunks cc ON cc.paragraph_id = p.paragraph_id
        WHERE  cc.chunk_id IS NULL
    ),
    para_quality AS (
        SELECT p2.paragraph_id, p2.doc_id, p2.full_text,
               COUNT(cc.chunk_id)     AS chunk_count,
               COUNT(ce.chunk_emb_id) AS emb_count
        FROM   ilpcsr_paragraphs p2
        JOIN   ilpcsr_content_chunks cc ON cc.paragraph_id = p2.paragraph_id
        LEFT JOIN ilpcsr_chunk_embeddings ce
               ON ce.chunk_id = cc.chunk_id AND ce.emb_model_id = %s
        GROUP BY p2.paragraph_id, p2.doc_id, p2.full_text
    )
    SELECT
        o.orphan_pid,
        o.doc_id,
        o.full_text,
        EXISTS (
            SELECT 1 FROM para_quality q
            WHERE  q.doc_id        = o.doc_id
              AND  q.full_text     = o.full_text          -- EXACT char-for-char
              AND  q.paragraph_id != o.orphan_pid
              AND  q.chunk_count   > 0
              AND  q.emb_count     = q.chunk_count        -- twin FULLY embedded
        ) AS has_fully_embedded_twin
    FROM orphan o
""", (EMB_MODEL_ID,))
rows = _cur.fetchall()
print(f"query done in {time.time()-t:.1f}s", flush=True)
_cur.close()
_conn.close()

total      = len(rows)
covered    = sum(1 for r in rows if r['has_fully_embedded_twin'])
uncovered  = [r for r in rows if not r['has_fully_embedded_twin']]

print("\n" + "=" * 70)
print("ORPHAN PARAGRAPH COVERAGE  (exact char-for-char twin, fully embedded)")
print("=" * 70)
print(f"  Orphan paragraphs (no chunks)            : {total}")
print(f"  With exact twin, fully chunked+embedded  : {covered}")
print(f"  WITHOUT such a twin ⚠                     : {len(uncovered)}")

if uncovered:
    print("\n  ⚠ orphans whose content is NOT yet fully covered by an embedded twin:")
    for r in uncovered[:20]:
        print(f"    doc={r['doc_id']}  text={repr((r['full_text'] or '')[:70])}")
    print("\n  (Likely these share text with one of the 30 zero-embedding CHECK-2 groups —")
    print("   embedding those 30 should bring this to 0. Re-run this cell afterward.)")
else:
    print("\n  PASS ✓ — every orphan paragraph's exact text is chunked + fully embedded via a twin.")

In [ ]:
import os, time, psycopg2, psycopg2.extras
from dotenv import load_dotenv
load_dotenv()

_conn = psycopg2.connect(
    dbname=os.environ['DB_NAME'], user=os.environ['DB_USER'],
    password=os.environ['DB_PASSWORD'], host=os.environ['DB_HOST'],
    port=os.environ['DB_PORT'], sslmode='require'
)
_conn.autocommit = True
_cur = _conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
_cur.execute("SET statement_timeout = '120s'")
EMB_MODEL_ID = 1
print("connected", flush=True)

# TRUE orphan debt at CONTENT grain:
# An orphan's content is unreachable ONLY if its (doc_id, full_text) contains some
# chunk_text whose every copy is unembedded — i.e. it intersects a CHECK-2 zero-emb group.
# A twin merely having an internal *duplicate* unembedded chunk does NOT count (content
# still embedded via the kept copy). This is the honest, content-grain coverage test.
t = time.time()
_cur.execute("""
    WITH zero_emb_fulltext AS (         -- (doc_id, full_text) that own an unembedded content piece
        SELECT p.doc_id, p.full_text
        FROM   ilpcsr_content_chunks cc
        JOIN   ilpcsr_paragraphs p ON p.paragraph_id = cc.paragraph_id
        LEFT JOIN ilpcsr_chunk_embeddings ce
               ON ce.chunk_id = cc.chunk_id AND ce.emb_model_id = %s
        GROUP  BY p.doc_id, p.full_text, cc.chunk_text
        HAVING COUNT(ce.chunk_emb_id) = 0
    ),
    orphan AS (                          -- paragraphs with no chunks
        SELECT p.paragraph_id, p.doc_id, p.full_text
        FROM   ilpcsr_paragraphs p
        LEFT JOIN ilpcsr_content_chunks cc ON cc.paragraph_id = p.paragraph_id
        WHERE  cc.chunk_id IS NULL
    )
    SELECT
        (SELECT COUNT(*) FROM orphan)                                   AS total_orphans,
        (SELECT COUNT(*) FROM orphan o
           WHERE EXISTS (SELECT 1 FROM zero_emb_fulltext z
                         WHERE z.doc_id = o.doc_id AND z.full_text = o.full_text)) AS truly_uncovered
""", (EMB_MODEL_ID,))
r = _cur.fetchone()
print(f"query done in {time.time()-t:.1f}s", flush=True)
_cur.close()
_conn.close()

total      = r['total_orphans']
uncovered  = r['truly_uncovered']
covered    = total - uncovered

print("\n" + "=" * 70)
print("ORPHAN COVERAGE — CONTENT GRAIN (the honest test)")
print("=" * 70)
print(f"  Orphan paragraphs (no chunks)                 : {total}")
print(f"  Content fully embedded elsewhere (covered)    : {covered}")
print(f"  Truly uncovered (text owns a CHECK-2 group) ⚠ : {uncovered}")
print()
if uncovered == 0:
    print("  PASS ✓ — all orphan content already embedded. Nothing to re-chunk.")
else:
    print(f"  {uncovered} orphans become covered the moment we embed the 30 CHECK-2 groups.")
    print("  → Fix the 30 (one INSERT each). Do NOT re-chunk the 711.")

In [ ]:
import os, time, psycopg2, psycopg2.extras
from dotenv import load_dotenv
load_dotenv()

_conn = psycopg2.connect(
    dbname=os.environ['DB_NAME'], user=os.environ['DB_USER'],
    password=os.environ['DB_PASSWORD'], host=os.environ['DB_HOST'],
    port=os.environ['DB_PORT'], sslmode='require'
)
_conn.autocommit = True
_cur = _conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
_cur.execute("SET statement_timeout = '180s'")
EMB_MODEL_ID = 1
print("connected", flush=True)

# Characterize the content groups behind the 95 truly-uncovered orphans.
# Per distinct chunk_text in a (doc_id, full_text) group: is it embedded anywhere?
# Then per group: how many distinct texts, how many are holes, how many embedded.
t = time.time()
_cur.execute("""
    WITH ct_level AS (
        SELECT p.doc_id, p.full_text, cc.chunk_text,
               COUNT(cc.chunk_id)     AS total_copies,
               COUNT(ce.chunk_emb_id) AS emb_copies
        FROM   ilpcsr_content_chunks cc
        JOIN   ilpcsr_paragraphs p ON p.paragraph_id = cc.paragraph_id
        LEFT JOIN ilpcsr_chunk_embeddings ce
               ON ce.chunk_id = cc.chunk_id AND ce.emb_model_id = %s
        GROUP  BY p.doc_id, p.full_text, cc.chunk_text
    ),
    grp AS (
        SELECT doc_id, full_text,
               COUNT(*)                                AS distinct_texts,
               COUNT(*) FILTER (WHERE emb_copies = 0)  AS hole_texts,
               COUNT(*) FILTER (WHERE emb_copies > 0)  AS embedded_texts
        FROM   ct_level
        GROUP  BY doc_id, full_text
    ),
    orphan AS (
        SELECT p.doc_id, p.full_text
        FROM   ilpcsr_paragraphs p
        LEFT JOIN ilpcsr_content_chunks cc ON cc.paragraph_id = p.paragraph_id
        WHERE  cc.chunk_id IS NULL
    ),
    affected AS (   -- content groups behind the truly-uncovered orphans
        SELECT DISTINCT g.doc_id, g.full_text, g.distinct_texts, g.hole_texts, g.embedded_texts
        FROM   orphan o
        JOIN   grp g ON g.doc_id = o.doc_id AND g.full_text = o.full_text
        WHERE  g.hole_texts > 0
    )
    SELECT
        COUNT(*)                                              AS affected_groups,
        COUNT(*) FILTER (WHERE embedded_texts > 0)           AS partially_embedded_groups,
        COUNT(*) FILTER (WHERE embedded_texts = 0)           AS fully_unembedded_groups,
        SUM(distinct_texts)                                  AS total_distinct_texts,
        SUM(hole_texts)                                      AS total_hole_texts,
        SUM(embedded_texts)                                  AS total_embedded_texts
    FROM affected
""", (EMB_MODEL_ID,))
r = _cur.fetchone()
print(f"query done in {time.time()-t:.1f}s", flush=True)
_cur.close()
_conn.close()

print("\n" + "=" * 70)
print("STATE OF THE CONTENT BEHIND THE 95 UNCOVERED ORPHANS")
print("=" * 70)
print(f"  Affected content groups                  : {r['affected_groups']}")
print(f"    → partially embedded (some holes)      : {r['partially_embedded_groups']}")
print(f"    → fully unembedded (every text a hole) : {r['fully_unembedded_groups']}")
print()
print(f"  Distinct chunk_texts in these groups     : {r['total_distinct_texts']}")
print(f"    → already embedded                     : {r['total_embedded_texts']}")
print(f"    → holes (need 1 embedding each)        : {r['total_hole_texts']}")
print()
print("  Verdict: all are CHUNKED (CHECK 1 = 0). None are unchunked.")
print("  The gap is purely missing chunk-embeddings = the holes above.")

# 9 - Chunk-level integrity (lossless chunking)

Confirms the **chunk text layer is complete**, independent of embeddings: every paragraph's
`full_text` is fully represented by its chunks, with nothing dropped during chunking.

**Method (deterministic re-derivation):** re-run the *same* `RecursiveCharacterTextSplitter`
(config from `ilpcsr_rag_config`) on each chunked paragraph's `full_text` in memory, and compare
the **set** of produced chunk texts to the **set** of `chunk_text` rows in the DB. They must be
equal -- the splitter is deterministic, and same-paragraph dedup only collapses exact duplicates
(which set-comparison ignores). Any mismatch is a real chunking defect.

Orphan paragraphs (no chunks) are skipped: their content is covered by an identical-`full_text`
twin (proven by Sec 8 CHECK 1), and that twin is checked here. Runs the splitter only -- no
embedding model, no GPU.

In [ ]:
import os, time, psycopg2
from dotenv import load_dotenv
try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except ImportError:
    from langchain.text_splitter import RecursiveCharacterTextSplitter
load_dotenv()

SAMPLE_LIMIT = None   # None = check every chunked paragraph; set an int to sample

_conn = psycopg2.connect(
    dbname=os.environ['DB_NAME'], user=os.environ['DB_USER'],
    password=os.environ['DB_PASSWORD'], host=os.environ['DB_HOST'],
    port=os.environ['DB_PORT'], sslmode='require'
)
_conn.autocommit = True
_cur = _conn.cursor()
_cur.execute("SET statement_timeout = '300s'")
EMB_MODEL_ID = 1

# Splitter config straight from the DB so it matches ingestion exactly.
_cur.execute("SELECT chunk_size, chunk_overlap FROM ilpcsr_rag_config WHERE emb_model_id = %s", (EMB_MODEL_ID,))
chunk_size, chunk_overlap = _cur.fetchone()
splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
print(f"splitter: chunk_size={chunk_size}, overlap={chunk_overlap}", flush=True)

# One row per chunked paragraph: its full_text + the set of chunk_texts the DB holds for it.
q = """
    SELECT p.paragraph_id, p.doc_id, p.full_text, array_agg(cc.chunk_text) AS db_chunks
    FROM   ilpcsr_paragraphs p
    JOIN   ilpcsr_content_chunks cc ON cc.paragraph_id = p.paragraph_id
    GROUP  BY p.paragraph_id, p.doc_id, p.full_text
"""
if SAMPLE_LIMIT:
    q += f" LIMIT {int(SAMPLE_LIMIT)}"

t = time.time()
_cur.execute(q)
rows = _cur.fetchall()
print(f"fetched {len(rows)} chunked paragraphs in {time.time()-t:.1f}s", flush=True)
_cur.close(); _conn.close()

mismatches = []
t = time.time()
for paragraph_id, doc_id, full_text, db_chunks in rows:
    db_set   = set(db_chunks)
    recomp   = set(splitter.split_text(full_text))
    if db_set != recomp:
        mismatches.append({
            "paragraph_id": paragraph_id, "doc_id": doc_id,
            "db_count": len(db_set), "recomputed_count": len(recomp),
            "missing_from_db":  sorted(recomp - db_set),   # text the splitter makes but DB lacks
            "extra_in_db":      sorted(db_set - recomp),    # text in DB the splitter never makes
        })
print(f"re-split + compared {len(rows)} paragraphs in {time.time()-t:.1f}s", flush=True)

print("\n" + "=" * 70)
print("CHUNK-LEVEL INTEGRITY (lossless chunking)")
print("=" * 70)
print(f"  Chunked paragraphs checked : {len(rows)}")
print(f"  Paragraphs with mismatch   : {len(mismatches)}")
if not mismatches:
    print("\n  PASS - every chunked paragraph's DB chunks == deterministic re-split.")
    print("  The chunk text layer is intact and lossless. Embedding gaps (if any) are")
    print("  a separate, embedding-only concern (see Sec 8 CHECK 2).")
else:
    print("\n  FAIL - these paragraphs' chunks do not match a clean re-split:")
    for m in mismatches[:15]:
        print(f"    para={m['paragraph_id']} doc={m['doc_id']} "
              f"db={m['db_count']} recomputed={m['recomputed_count']}")
        if m['missing_from_db']:
            print(f"      missing_from_db[0]={repr(m['missing_from_db'][0][:80])}")
        if m['extra_in_db']:
            print(f"      extra_in_db[0]={repr(m['extra_in_db'][0][:80])}")


## After running `embed_missing_chunks.py`

`ingestion/embed_missing_chunks.py` (v2) embeds one representative chunk per zero-embedding content
group. After running it, re-run **Sec 8 CHECK 2** (`f4248217`) and the coverage cell (`cc4e631d`):
both should read **0**. Combined with Sec 9 PASS, the corpus is in the canonical end state -
every distinct piece of content is chunked and embedded at least once.